# 第12章: 脚式オドメトリ - 脚ロボットのキネマティクス

**SLAM Handbook Chapter 12, Section 12.1-12.3**

このノートブックでは、脚式ロボットのオドメトリとSLAMへの応用の基礎を学びます。

## 学習内容
1. 2リンク平面脚の順運動学（Forward Kinematics）
2. 接地検出の概念
3. 順運動学 + 接地情報を用いた脚オドメトリ
4. キネマティクスファクターを用いたファクターグラフ

車輪ロボットと異なり、脚式ロボットは不整地を歩行できますが、スリップや地形の不確実性によりオドメトリの推定が複雑になります。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.collections import LineCollection

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 12.1 2リンク平面脚の順運動学

**SLAM Handbook Section 12.1**

最も基本的な脚モデルとして、2リンク平面脚を考えます。

### 順運動学（Forward Kinematics）
ボディ座標原点（股関節）からの足先位置：

$$x_{foot} = l_1 \cos\theta_1 + l_2 \cos(\theta_1 + \theta_2)$$
$$y_{foot} = l_1 \sin\theta_1 + l_2 \sin(\theta_1 + \theta_2)$$

ここで：
- $l_1, l_2$: 上腿・下腿のリンク長
- $\theta_1$: 股関節角度（ボディに対する）
- $\theta_2$: 膝関節角度（上腿に対する）

### ヤコビアン
$$\mathbf{J} = \begin{bmatrix} -l_1 s_1 - l_2 s_{12} & -l_2 s_{12} \\ l_1 c_1 + l_2 c_{12} & l_2 c_{12} \end{bmatrix}$$

ここで $s_1 = \sin\theta_1$, $c_1 = \cos\theta_1$, $s_{12} = \sin(\theta_1+\theta_2)$, $c_{12} = \cos(\theta_1+\theta_2)$

In [ ]:
class PlanarLeg2Link:
    """2-link planar leg model."""

    def __init__(self, l1=0.3, l2=0.3):
        self.l1 = l1  # Upper leg length [m]
        self.l2 = l2  # Lower leg length [m]

    def forward_kinematics(self, theta1, theta2):
        """Compute foot position in body frame (hip at origin)."""
        x_knee = self.l1 * np.cos(theta1)
        y_knee = self.l1 * np.sin(theta1)
        x_foot = x_knee + self.l2 * np.cos(theta1 + theta2)
        y_foot = y_knee + self.l2 * np.sin(theta1 + theta2)
        return np.array([x_foot, y_foot]), np.array([x_knee, y_knee])

    def jacobian(self, theta1, theta2):
        """Compute the Jacobian matrix."""
        s1 = np.sin(theta1)
        c1 = np.cos(theta1)
        s12 = np.sin(theta1 + theta2)
        c12 = np.cos(theta1 + theta2)
        J = np.array([
            [-self.l1 * s1 - self.l2 * s12, -self.l2 * s12],
            [self.l1 * c1 + self.l2 * c12, self.l2 * c12]
        ])
        return J

    def draw(self, ax, body_pos, theta1, theta2, color='blue', label=None):
        """Draw the leg configuration."""
        foot, knee = self.forward_kinematics(theta1, theta2)
        hip = body_pos
        knee_world = hip + knee
        foot_world = hip + foot

        # Draw links
        ax.plot([hip[0], knee_world[0]], [hip[1], knee_world[1]],
                color=color, linewidth=3, solid_capstyle='round')
        ax.plot([knee_world[0], foot_world[0]], [knee_world[1], foot_world[1]],
                color=color, linewidth=3, solid_capstyle='round')
        # Draw joints
        ax.plot(*hip, 'o', color=color, markersize=8)
        ax.plot(*knee_world, 'o', color=color, markersize=6)
        ax.plot(*foot_world, 's', color=color, markersize=6, label=label)

        return foot_world

# Demonstrate forward kinematics
leg = PlanarLeg2Link(l1=0.3, l2=0.3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Various leg configurations
ax = axes[0]
configs = [
    (-np.pi/4, -np.pi/3, 'blue', '$\\theta_1=-45, \\theta_2=-60$'),
    (-np.pi/6, -np.pi/2, 'red', '$\\theta_1=-30, \\theta_2=-90$'),
    (-np.pi/3, -np.pi/6, 'green', '$\\theta_1=-60, \\theta_2=-30$'),
]
for theta1, theta2, color, label in configs:
    leg.draw(ax, np.array([0, 0.6]), theta1, theta2, color=color, label=label)
ax.set_xlim(-0.4, 0.4)
ax.set_ylim(-0.15, 0.75)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
ax.set_title('Leg Configurations\n(hip at top)')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

# Plot 2: Workspace (reachable area)
ax = axes[1]
theta1_range = np.linspace(-np.pi, 0, 100)
theta2_range = np.linspace(-np.pi, 0, 100)
workspace_x = []
workspace_y = []
for t1 in theta1_range:
    for t2 in theta2_range:
        foot, _ = leg.forward_kinematics(t1, t2)
        workspace_x.append(foot[0])
        workspace_y.append(foot[1])
ax.scatter(workspace_x, workspace_y, s=0.2, alpha=0.3, c='blue')
ax.plot(0, 0, 'r^', markersize=10, label='Hip joint')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title('Leg Workspace\n(reachable foot positions)')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

# Plot 3: Jacobian - velocity mapping
ax = axes[2]
theta1, theta2 = -np.pi/4, -np.pi/2
foot, knee = leg.forward_kinematics(theta1, theta2)
J = leg.jacobian(theta1, theta2)

# Draw leg
body = np.array([0, 0.6])
leg.draw(ax, body, theta1, theta2, color='gray')

# Draw velocity ellipse at foot
angles_ell = np.linspace(0, 2*np.pi, 100)
unit_circle = np.column_stack([np.cos(angles_ell), np.sin(angles_ell)])
# Scale for visibility
scale = 0.1
ellipse_pts = (J @ unit_circle.T * scale).T
foot_world = body + foot
ax.plot(foot_world[0] + ellipse_pts[:, 0], foot_world[1] + ellipse_pts[:, 1],
        'r-', linewidth=2, label='Velocity ellipse')
ax.set_xlim(-0.4, 0.4)
ax.set_ylim(-0.15, 0.75)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title('Jacobian Velocity Ellipse\nat foot position')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

plt.tight_layout()
plt.show()

## 12.2 接地検出と脚オドメトリ

**SLAM Handbook Section 12.2**

脚オドメトリの基本原理：
1. **接地検出**: 足が地面に接触しているかを判定（力センサー、接触スイッチなど）
2. **接地拘束**: 接地中の足先は地面に対して静止（スリップなし仮定）
3. **ボディ運動推定**: 接地足を基準点として、関節角度の変化からボディの移動を推定

$$\mathbf{p}_{body}(t+dt) = \mathbf{p}_{foot}^{world} - FK(\boldsymbol{\theta}(t+dt))$$

ここで $\mathbf{p}_{foot}^{world}$ は接地中の足先のワールド座標（固定）、$FK$ は順運動学です。

In [ ]:
def simulate_walking(leg, n_steps=200, dt=0.01, stride_length=0.15, body_height=0.5):
    """
    Simulate planar walking with a 2-link leg.
    Simplified model: body moves forward, leg swings in a gait pattern.
    """
    t_array = np.arange(n_steps) * dt
    gait_period = 0.5  # seconds per step

    # Ground truth body position
    body_speed = stride_length / gait_period  # m/s
    body_x_true = body_speed * t_array
    body_y_true = body_height * np.ones(n_steps)

    # Joint angles (sinusoidal gait pattern)
    phase = 2 * np.pi * t_array / gait_period
    theta1 = -np.pi/4 + 0.2 * np.sin(phase)
    theta2 = -np.pi/3 + 0.15 * np.sin(phase + np.pi/3)

    # Contact state: foot is on ground during stance phase
    contact = np.sin(phase) < 0.3  # roughly 60% stance phase

    # Add joint angle noise (encoder noise)
    joint_noise = 0.005  # rad
    theta1_noisy = theta1 + np.random.normal(0, joint_noise, n_steps)
    theta2_noisy = theta2 + np.random.normal(0, joint_noise, n_steps)

    return (t_array, body_x_true, body_y_true, theta1, theta2,
            theta1_noisy, theta2_noisy, contact)

def leg_odometry(leg, body_x_true, body_y_true, theta1, theta2, contact):
    """
    Estimate body trajectory using leg odometry.
    During contact: body_pos = foot_world - FK(theta)
    """
    n = len(theta1)
    body_x_est = np.zeros(n)
    body_y_est = np.zeros(n)

    # Initialize with true first position
    body_x_est[0] = body_x_true[0]
    body_y_est[0] = body_y_true[0]

    foot_world = None  # World-frame foot position (fixed during contact)

    for i in range(1, n):
        foot_body, _ = leg.forward_kinematics(theta1[i], theta2[i])

        if contact[i]:
            if not contact[i-1] or foot_world is None:
                # New contact: compute foot world position from current estimate
                foot_world = np.array([body_x_est[i-1], body_y_est[i-1]]) + foot_body

            # Contact: estimate body from fixed foot
            body_x_est[i] = foot_world[0] - foot_body[0]
            body_y_est[i] = foot_world[1] - foot_body[1]
        else:
            # Swing phase: use simple forward prediction
            body_x_est[i] = body_x_est[i-1] + (body_x_est[i-1] - body_x_est[max(0,i-2)])
            body_y_est[i] = body_y_est[i-1]

    return body_x_est, body_y_est

# Simulate walking
(t_walk, bx_true, by_true, th1, th2,
 th1_noisy, th2_noisy, contact) = simulate_walking(leg, n_steps=300)

# Compute leg odometry with clean and noisy joints
bx_est_clean, by_est_clean = leg_odometry(leg, bx_true, by_true, th1, th2, contact)
bx_est_noisy, by_est_noisy = leg_odometry(leg, bx_true, by_true, th1_noisy, th2_noisy, contact)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Walking animation snapshots
ax = axes[0, 0]
snapshot_indices = np.linspace(0, len(t_walk)-1, 8, dtype=int)
for idx in snapshot_indices:
    body_pos = np.array([bx_true[idx], by_true[idx]])
    leg.draw(ax, body_pos, th1[idx], th2[idx],
             color='blue' if contact[idx] else 'red')
    # Draw body as rectangle
    ax.plot([body_pos[0]-0.05, body_pos[0]+0.05],
            [body_pos[1], body_pos[1]], 'k-', linewidth=5)
ax.set_xlim(-0.1, 1.2)
ax.set_ylim(-0.1, 0.7)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Walking Gait Snapshots\n(blue=stance, red=swing)')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

# Joint angles and contact
ax = axes[0, 1]
ax.plot(t_walk, np.degrees(th1), label='$\\theta_1$ (hip)')
ax.plot(t_walk, np.degrees(th2), label='$\\theta_2$ (knee)')
ax2 = ax.twinx()
ax2.fill_between(t_walk, 0, contact.astype(float), alpha=0.2, color='green', label='Contact')
ax2.set_ylabel('Contact state')
ax2.set_ylim(-0.1, 1.5)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Joint angle [deg]')
ax.set_title('Joint Angles & Contact State')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Body trajectory comparison
ax = axes[1, 0]
ax.plot(t_walk, bx_true, 'b-', linewidth=2, label='Ground Truth')
ax.plot(t_walk, bx_est_clean, 'g--', label='Leg Odometry (clean)')
ax.plot(t_walk, bx_est_noisy, 'r--', alpha=0.7, label='Leg Odometry (noisy)')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Body x position [m]')
ax.set_title('Body X Position Over Time')
ax.legend()
ax.grid(True, alpha=0.3)

# Position error
ax = axes[1, 1]
error_clean = np.abs(bx_est_clean - bx_true)
error_noisy = np.abs(bx_est_noisy - bx_true)
ax.plot(t_walk, error_clean, 'g-', label=f'Clean (final: {error_clean[-1]:.4f} m)')
ax.plot(t_walk, error_noisy, 'r-', alpha=0.7, label=f'Noisy (final: {error_noisy[-1]:.4f} m)')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Position error [m]')
ax.set_title('Leg Odometry Error')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12.3 ファクターグラフとキネマティクスファクター

**SLAM Handbook Section 12.3**

脚ロボットのSLAMでは、以下のファクターをファクターグラフに組み込みます：

1. **IMUプリインテグレーションファクター**: ボディの慣性運動制約
2. **キネマティクスファクター**: 接地足とボディ間の幾何学的制約
   $$\mathbf{r}_{kin} = \mathbf{p}_{foot}^{world} - (\mathbf{p}_{body} + \mathbf{R}_{body} \cdot FK(\boldsymbol{\theta}))$$
3. **接地制約ファクター**: 接地中の足先速度ゼロ制約

以下では、これらのファクターを含む簡易ファクターグラフを可視化します。

In [ ]:
# Factor graph visualization for legged robot SLAM
fig, ax = plt.subplots(1, 1, figsize=(14, 7))

n_keyframes = 6
kf_x = np.linspace(0.5, 5.5, n_keyframes)
kf_y = np.ones(n_keyframes) * 4.0

# Foot contact positions (alternating feet)
foot_x = np.linspace(0.8, 5.2, n_keyframes)
foot_y_left = np.ones(n_keyframes) * 2.0
foot_y_right = np.ones(n_keyframes) * 1.0

# Draw body pose nodes
for i in range(n_keyframes):
    ax.plot(kf_x[i], kf_y[i], 'ko', markersize=18, zorder=5)
    ax.plot(kf_x[i], kf_y[i], 'co', markersize=14, zorder=6)
    ax.text(kf_x[i], kf_y[i] + 0.3, f'$x_{i}$', ha='center', fontsize=11)

# Draw IMU factors (between body poses)
for i in range(n_keyframes - 1):
    ax.annotate('', xy=(kf_x[i+1], kf_y[i+1]), xytext=(kf_x[i], kf_y[i]),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.5))

# Draw foot position nodes and kinematic factors
for i in range(n_keyframes):
    if i % 2 == 0:  # Left foot contact
        fy = foot_y_left[i]
        fc = 'blue'
        label = 'Left foot' if i == 0 else None
    else:
        fy = foot_y_right[i]
        fc = 'green'
        label = 'Right foot' if i == 1 else None

    # Foot node
    ax.plot(foot_x[i], fy, 's', color=fc, markersize=12, zorder=5, label=label)
    ax.text(foot_x[i], fy - 0.35, f'$f_{i}$', ha='center', fontsize=10)

    # Kinematic factor (body -> foot)
    ax.plot([kf_x[i], foot_x[i]], [kf_y[i], fy], '--', color='purple',
            linewidth=1.5, alpha=0.7)
    # Factor node (small square)
    mid_x = (kf_x[i] + foot_x[i]) / 2
    mid_y = (kf_y[i] + fy) / 2
    ax.plot(mid_x, mid_y, 'ks', markersize=6, zorder=5)

# Draw contact constraints (between consecutive foot positions of same foot)
for i in range(0, n_keyframes - 2, 2):
    ax.plot([foot_x[i], foot_x[i+2]], [foot_y_left[i], foot_y_left[i+2]],
            ':', color='blue', linewidth=1.5, alpha=0.5)
for i in range(1, n_keyframes - 2, 2):
    ax.plot([foot_x[i], foot_x[i+2]], [foot_y_right[i], foot_y_right[i+2]],
            ':', color='green', linewidth=1.5, alpha=0.5)

# Prior factor
ax.plot(kf_x[0] - 0.3, kf_y[0], 'r^', markersize=12, zorder=5)
ax.plot([kf_x[0] - 0.3, kf_x[0]], [kf_y[0], kf_y[0]], 'r-', linewidth=2)

# Ground line
ax.axhline(y=0.5, color='brown', linewidth=3, alpha=0.3)
ax.text(3, 0.2, 'Ground plane', ha='center', fontsize=10, color='brown')

# Legend
ax.plot([], [], 'r->', linewidth=2.5, label='IMU preintegration factor')
ax.plot([], [], '--', color='purple', linewidth=1.5, label='Kinematic factor (FK)')
ax.plot([], [], ':', color='gray', linewidth=1.5, label='Contact constraint')
ax.plot([], [], 'r^', markersize=10, label='Prior factor')
ax.plot([], [], 'co', markersize=12, label='Body pose node')

ax.set_xlim(-0.2, 6.5)
ax.set_ylim(-0.2, 5.5)
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=10, ncol=2)
ax.set_title('Factor Graph for Legged Robot SLAM\n脚ロボットSLAMのファクターグラフ', fontsize=13)
ax.grid(True, alpha=0.2)
ax.set_xlabel('x')
ax.set_ylabel('Graph layout')

plt.tight_layout()
plt.show()

## 演習問題

1. **逆運動学**: 与えられた足先位置 $(x, y)$ から関節角度 $(\theta_1, \theta_2)$ を解析的に求める逆運動学を実装せよ。
2. **スリップ検出**: 接地中に足先が滑った場合（足先位置が変化）を検出するアルゴリズムを考え、実装せよ。ヒント：順運動学から推定されるボディ速度とIMU速度の不一致を利用。
3. **4足歩行**: 2本の脚を持つモデルに拡張し、左右の脚が交互に接地する歩行パターンを実装せよ。複数の接地足からのオドメトリ推定をどう融合するか考察せよ。

## まとめ

本ノートブックでは以下を学びました：

1. **2リンク平面脚の順運動学**: 関節角度から足先位置を計算する基本的な運動学モデルとヤコビアン
2. **接地検出と脚オドメトリ**: 接地中の足先を固定点として利用し、関節角度の変化からボディの移動を推定
3. **ファクターグラフ統合**: キネマティクスファクター、IMUファクター、接地拘束ファクターの組み合わせ

脚式オドメトリは、IMU、LiDAR、カメラなど他のセンサーと融合することで、不整地環境での堅牢な状態推定を実現します。特にSpot、ANYmalなどの四足歩行ロボットにおいて実用化が進んでいます。

**参考**: SLAM Handbook Chapter 12